# RNN 实战：回归网格搜索与情感分类

本 Notebook 包含两部分内容：
1. **RNN 回归模型网格搜索**：使用 RNN 预测正弦波，并通过网格搜索寻找最佳超参数（hidden_size, learning_rate），结合早停机制防止过拟合。
2. **情感分类实战**：从零构建 Tokenizer（分词、词典构建、编码解码），实现 Dataset 与 DataLoader（动态 padding），并搭建包含 Embedding 层的 RNN 模型进行文本分类。

In [1]:
# 导入 PyTorch 深度学习框架
import torch
import torch.nn as nn
import torch.optim as optim

# 导入数值计算和绘图库
import numpy as np
import matplotlib.pyplot as plt

# 导入中文分词库 jieba
import jieba

# 导入 PyTorch 数据处理工具
from torch.utils.data import Dataset, DataLoader

# 导入集合工具，用于词频统计
from collections import Counter
# 导入迭代器工具，用于网格搜索参数组合
import itertools

# 设置随机种子以保证结果可复现
# 深度学习中很多初始化是随机的，固定种子可以让每次运行结果一致
torch.manual_seed(42)
np.random.seed(42)

# 检查设备，优先使用 GPU (cuda)，否则使用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 一、RNN 回归模型网格搜索

### 1. 数据生成
生成正弦波数据作为回归任务的数据集。任务是根据前 `seq_len` 个点预测下一个点的值。

In [2]:
def generate_sine_data(seq_len=20, num_samples=1000):
    """
    生成正弦波数据用于回归任务
    参数:
        seq_len: 序列长度，即用多少个历史点来预测下一个点
        num_samples: 样本总数
    返回:
        X: 输入数据，形状 (num_samples, seq_len, 1)
        y: 标签数据，形状 (num_samples, 1)
    """
    X_list = []
    y_list = []
    
    # 循环生成 num_samples 个样本
    for _ in range(num_samples):
        # 随机选择一个起始点，范围在 [0, 2π]
        start = np.random.rand() * 2 * np.pi
        
        # 生成一段正弦波序列
        # linspace 生成从 start 到 start+4π 的等间距点
        # 长度为 seq_len + 1，前 seq_len 个作为输入，最后一个作为预测目标
        x_seq = np.sin(np.linspace(start, start + 4 * np.pi, seq_len + 1))
        
        # 将前 seq_len 个点作为输入特征 X
        X_list.append(x_seq[:-1]) 
        # 将最后一个点作为目标标签 y
        y_list.append(x_seq[-1])
    
    # 将列表转换为 NumPy 数组，再转换为 PyTorch 张量
    # unsqueeze(-1) 是为了增加特征维度
    # RNN 要求输入形状为 (batch_size, seq_len, input_size)
    # 这里的 input_size 为 1（单变量时间序列），所以需要扩展最后一维
    X = torch.tensor(np.array(X_list), dtype=torch.float32).unsqueeze(-1)
    y = torch.tensor(np.array(y_list), dtype=torch.float32).unsqueeze(-1)
    
    return X, y

# 调用函数生成数据
X, y = generate_sine_data()

# 划分训练集和验证集 (8:2 比例)
train_size = int(0.8 * len(X))

# 切片划分数据，并将数据移动到计算设备 (CPU 或 GPU)
X_train, X_val = X[:train_size].to(device), X[train_size:].to(device)
y_train, y_val = y[:train_size].to(device), y[train_size:].to(device)

# 打印数据集形状确认
print(f"训练集: {X_train.shape}, 验证集: {X_val.shape}")

训练集: torch.Size([800, 20, 1]), 验证集: torch.Size([200, 20, 1])


### 2. 定义 RNN 回归模型
- 输入维度 `input_size=1` (单变量时间序列)
- 输出维度 `output_size=1` (预测一个标量值)
- 包含一个 RNN 层和一个全连接层

In [3]:
class RNNRegression(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNRegression, self).__init__()
        # 保存隐藏层大小
        self.hidden_size = hidden_size
        
        # 定义 RNN 层
        # input_size: 输入特征维度 (这里是 1)
        # hidden_size: 隐藏层神经元数量 (网格搜索会调整这个参数)
        # batch_first=True: 指定输入数据的第一个维度是 batch_size，即 (batch, seq, feature)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        
        # 定义全连接层，用于将 RNN 的输出映射到最终的预测值
        # 输入是 hidden_size，输出是 output_size (这里是 1)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x 的形状: (batch_size, seq_len, input_size)
        
        # RNN 前向传播
        # out: 包含所有时间步的输出，形状 (batch_size, seq_len, hidden_size)
        # h_n: 最后一个时间步的隐藏状态，形状 (num_layers, batch_size, hidden_size)
        out, h_n = self.rnn(x)
        
        # 取最后一个时间步的输出用于预测
        # out[:, -1, :] 表示取所有样本 (:) 的最后一个时间步 (-1) 的所有特征 (:)
        # 形状变为 (batch_size, hidden_size)
        last_step_out = out[:, -1, :]
        
        # 通过全连接层得到最终预测值
        # 形状变为 (batch_size, output_size)
        prediction = self.fc(last_step_out)
        
        return prediction

### 3. 网格搜索与早停
- **网格搜索**：通过双重循环遍历 `hidden_size` 和 `learning_rate` 的组合。
- **早停 (Early Stopping)**：监控验证集损失，如果连续 `patience` 个 epoch 没有下降，则停止训练。
- **模型初始化**：每次网格搜索迭代时，重新实例化模型和优化器。

In [4]:
# 定义超参数网格 (Grid Search)
# 我们将尝试 hidden_size 和 lr 的所有组合
param_grid = {
    'hidden_size': [16, 32, 64],  # 尝试不同的隐藏层大小
    'lr': [0.01, 0.001]           # 尝试不同的学习率
}

# 训练配置
epochs = 200      # 最大训练轮数
patience = 10     # 早停忍耐轮数：如果验证集 loss 连续 10 轮不下降，则停止训练

# 记录全局最佳结果的变量
best_loss = float('inf')  # 初始化为无穷大
best_params = None        # 最佳参数组合
best_model_state = None   # 最佳模型权重

# 生成所有参数组合
# param_grid.items() 返回 [('hidden_size', [16, 32, 64]), ('lr', [0.01, 0.001])]
# zip(*...) 解包为 keys=('hidden_size', 'lr'), values=([16, 32, 64], [0.01, 0.001])
keys, values = zip(*param_grid.items())
# itertools.product(*values) 生成笛卡尔积，即所有组合
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"开始网格搜索，共 {len(combinations)} 组参数...")

# 遍历每一组参数组合
for i, params in enumerate(combinations):
    print(f"\n[{i+1}/{len(combinations)}] 测试参数: {params}")
    
    # 1. 模型初始化 (重要：每次网格搜索必须重新初始化模型)
    # 否则模型会继承上一次循环的参数，导致结果不准确
    model = RNNRegression(input_size=1, hidden_size=params['hidden_size'], output_size=1).to(device)
    
    # 2. 初始化优化器和损失函数
    # 优化器需要传入当前模型的参数
    optimizer = optim.Adam(model.parameters(), lr=params['lr'])
    criterion = nn.MSELoss() # 回归任务通常使用均方误差损失
    
    # 当前参数组合下的早停变量
    min_val_loss = float('inf') # 记录当前组合的最小验证损失
    no_improve_epochs = 0       # 记录连续未提升的轮数
    
    # 开始训练循环
    for epoch in range(epochs):
        # --- 训练阶段 ---
        model.train()           # 设置为训练模式
        optimizer.zero_grad()   # 清空梯度
        
        outputs = model(X_train)       # 前向传播
        loss = criterion(outputs, y_train) # 计算训练损失
        
        loss.backward()         # 反向传播
        optimizer.step()        # 更新参数
        
        # --- 验证阶段 ---
        model.eval()            # 设置为评估模式
        with torch.no_grad():   # 不计算梯度，节省内存
            val_outputs = model(X_val)            # 验证集前向传播
            val_loss = criterion(val_outputs, y_val).item() # 计算验证损失
        
        # --- 早停检查 ---
        if val_loss < min_val_loss:
            min_val_loss = val_loss   # 更新最小损失
            no_improve_epochs = 0     # 重置计数器
        else:
            no_improve_epochs += 1    # 计数器加 1
            
        # 如果超过耐心值，触发早停
        if no_improve_epochs >= patience:
            print(f"  早停触发于 epoch {epoch+1}, 最佳验证损失: {min_val_loss:.6f}")
            break
    
    # --- 记录全局最佳结果 ---
    if min_val_loss < best_loss:
        best_loss = min_val_loss
        best_params = params
        best_model_state = model.state_dict() # 保存模型权重
        print(f"  --> 发现新最佳参数! loss: {best_loss:.6f}")

print("="*40)
print(f"网格搜索完成。\n最佳参数: {best_params}\n最佳验证损失: {best_loss:.6f}")

开始网格搜索，共 6 组参数...

[1/6] 测试参数: {'hidden_size': 16, 'lr': 0.01}
  --> 发现新最佳参数! loss: 0.000008

[2/6] 测试参数: {'hidden_size': 16, 'lr': 0.001}
  早停触发于 epoch 69, 最佳验证损失: 0.001031

[3/6] 测试参数: {'hidden_size': 32, 'lr': 0.01}
  早停触发于 epoch 176, 最佳验证损失: 0.000000
  --> 发现新最佳参数! loss: 0.000000

[4/6] 测试参数: {'hidden_size': 32, 'lr': 0.001}
  早停触发于 epoch 49, 最佳验证损失: 0.002619

[5/6] 测试参数: {'hidden_size': 64, 'lr': 0.01}
  早停触发于 epoch 13, 最佳验证损失: 0.006485

[6/6] 测试参数: {'hidden_size': 64, 'lr': 0.001}
  早停触发于 epoch 41, 最佳验证损失: 0.000757
网格搜索完成。
最佳参数: {'hidden_size': 32, 'lr': 0.01}
最佳验证损失: 0.000000


## 二、情感分类实战

### 1. 模拟数据集与 Tokenizer
我们首先定义一个简单的中文情感分类数据集，然后实现 `Tokenizer` 类。

**Tokenizer 功能要求：**
1. **分词**：支持 `jieba` 分词 (word-level)。
2. **构建词典**：统计词频，支持特殊 token (`<pad>`, `<unk>`, `<bos>`, `<eos>`)。
3. **Encode**：Batch 文本 -> ID 序列（自动添加 `<bos>`, `<eos>`）。
4. **Decode**：Batch ID 序列 -> 文本。
5. **统计最大长度**：用于 Dataset 分析。

In [5]:
class Tokenizer:
    def __init__(self, vocab=None, max_len=None):
        self.max_len = max_len
        self.word2idx = {} # 单词到索引的映射
        self.idx2word = {} # 索引到单词的映射
        
        # 定义特殊 token
        self.pad_token = "<pad>" # 填充 token，用于补齐序列长度
        self.unk_token = "<unk>" # 未知词 token，用于处理词表中不存在的词
        self.bos_token = "<bos>" # 序列开始 token (Begin of Sequence)
        self.eos_token = "<eos>" # 序列结束 token (End of Sequence)
        
        # 初始化特殊 token 的 ID
        self.pad_id = 0
        self.unk_id = 1
        self.bos_id = 2
        self.eos_id = 3
        
        # 初始化词表
        if vocab:
            self.word2idx = vocab
            self.idx2word = {v: k for k, v in vocab.items()}
        else:
            # 如果没有传入词表，初始化只包含特殊 token 的基础词表
            self.word2idx = {
                self.pad_token: self.pad_id,
                self.unk_token: self.unk_id,
                self.bos_token: self.bos_id,
                self.eos_token: self.eos_id
            }
            self.idx2word = {v: k for k, v in self.word2idx.items()}

    def fit_on_texts(self, texts):
        """
        根据文本构建词典
        1. 使用 jieba 分词
        2. 统计词频
        3. 将高频词加入词典
        """
        all_words = []
        max_len_stat = 0 # 统计样本的最大长度
        
        for text in texts:
            # 使用 jieba 进行中文分词
            words = list(jieba.cut(text))
            all_words.extend(words)
            # 更新最大长度记录
            max_len_stat = max(max_len_stat, len(words))
        
        # 统计词频
        word_counts = Counter(all_words)
        # 按词频从高到低排序
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        
        # 将排序后的词加入词典
        curr_idx = len(self.word2idx) # 从当前词典长度开始编号
        for word, _ in sorted_words:
            if word not in self.word2idx:
                self.word2idx[word] = curr_idx
                self.idx2word[curr_idx] = word
                curr_idx += 1
        
        print(f"词典构建完成，词汇量: {len(self.word2idx)}, 样本最大长度: {max_len_stat}")
        return self

    def encode(self, texts, add_special_tokens=True):
        """
        将文本列表编码为 ID 列表
        texts: 文本列表 ["文本1", "文本2", ...]
        返回: [[id1, id2, ...], [...]]
        """
        batch_ids = []
        for text in texts:
            # 1. 分词
            words = list(jieba.cut(text))
            
            # 2. 查表转 ID，如果词不在词典中，使用 unk_id
            ids = [self.word2idx.get(w, self.unk_id) for w in words]
            
            # 3. 添加首尾特殊 token
            if add_special_tokens:
                ids = [self.bos_id] + ids + [self.eos_id]
            
            batch_ids.append(ids)
        return batch_ids

    def decode(self, batch_ids):
        """
        将 ID 列表解码为文本
        batch_ids: [[id1, ...], ...]
        """
        texts = []
        for ids in batch_ids:
            words = []
            for idx in ids:
                # 跳过特殊 token (pad, bos, eos)，只显示实际内容
                if idx in [self.pad_id, self.bos_id, self.eos_id]:
                    continue
                # 查表转词，如果是未知 ID，显示 <unk>
                words.append(self.idx2word.get(idx, self.unk_token))
            # 拼接成字符串
            texts.append("".join(words))
        return texts

    def __len__(self):
        # 返回词典大小
        return len(self.word2idx)

### 2. Dataset 与 DataLoader
- **Dataset**: 简单的 Map-style 数据集，返回 `(text, label)`。
- **collate_fn**: 
    1. 接收一个 batch 的样本列表。
    2. 调用 `tokenizer.encode` 将文本转为 ID。
    3. 获取当前 batch 的最大长度 (`seq_len`)。
    4. 使用 `<pad>` 对短序列进行填充。
    5. 转换为 Tensor 并返回。

In [6]:
# 模拟情感分类数据 (文本, 标签)
# 1 表示正面情感，0 表示负面情感
data = [
    ("这部电影太好看了，剧情紧凑", 1),
    ("服务员态度很差，再也不来了", 0),
    ("产品质量不错，物流也快", 1),
    ("由于包装破损，体验很不好", 0),
    ("这真是一个惊喜，非常喜欢", 1),
    ("味道一般，不推荐", 0),
    ("强烈推荐大家去看看", 1),
    ("浪费时间，毫无逻辑", 0)
]
# 提取文本和标签列表
texts = [d[0] for d in data]
labels = [d[1] for d in data]

# 实例化并训练 Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

# 定义 Dataset 类
class SentimentDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # 返回单条样本 (text, label)
        return self.data[idx]

# 自定义 collate_fn 函数
# DataLoader 在从 Dataset 取出一个 batch 的数据后，会调用这个函数进行处理
def collate_fn(batch):
    """
    处理变长序列填充 (Padding)
    batch: 一个列表，包含 batch_size 个 (text, label) 元组
    """
    # 解包 batch，分离文本和标签
    texts = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    
    # 1. 编码
    # 调用 tokenizer 将文本转换为 ID 列表
    # 结果 batch_ids 是一个列表，其中每个元素是一个 ID 列表，长度可能不同
    batch_ids = tokenizer.encode(texts)
    
    # 2. 获取当前 batch 的最大长度
    # 找出当前 batch 中最长的那个序列的长度
    max_len_in_batch = max(len(ids) for ids in batch_ids)
    
    # 3. Padding (填充)
    padded_ids = []
    for ids in batch_ids:
        # 计算需要填充的数量
        pad_num = max_len_in_batch - len(ids)
        # 在序列后面填充 pad_id，使其长度等于 max_len_in_batch
        padded_ids.append(ids + [tokenizer.pad_id] * pad_num)
    
    # 4. 转换为 Tensor
    # 输入特征 X 需要是 LongTensor (整数)
    X = torch.tensor(padded_ids, dtype=torch.long)
    # 标签 y 需要是 LongTensor (用于 CrossEntropyLoss)
    y = torch.tensor(labels, dtype=torch.long)
    
    return X, y

# 创建 DataLoader
dataset = SentimentDataset(data)
# batch_size=2: 每次取 2 个样本
# shuffle=True: 每个 epoch 打乱顺序
# collate_fn=collate_fn: 使用自定义的填充函数
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

# 测试 DataLoader
print("DataLoader 测试:")
for X_batch, y_batch in dataloader:
    print(f"X shape: {X_batch.shape}, y shape: {y_batch.shape}")
    # 验证解码结果，查看 padding 是否生效
    print("解码第一个样本:", tokenizer.decode(X_batch[0].unsqueeze(0).tolist())[0])
    break # 只打印一个 batch

Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/4n/s9lq2sbx5m18fjvq1r7w3hfr0000gn/T/jieba.cache
Loading model cost 0.406 seconds.
Prefix dict has been built successfully.


词典构建完成，词汇量: 45, 样本最大长度: 8
DataLoader 测试:
X shape: torch.Size([2, 9]), y shape: torch.Size([2])
解码第一个样本: 浪费时间，毫无逻辑


### 3. 搭建情感分类模型
模型结构：
1. **Embedding 层**：将离散的 ID 转换为稠密向量。`nn.Embedding(vocab_size, embedding_dim)`。
    - 这里的 Embedding 就像一个查表操作，输入的 ID 是多少，就取出矩阵中第几行的向量。
2. **RNN 层**：处理序列信息。
3. **全连接层**：将 RNN 最后一个时间步的输出映射到类别空间 (二分类)。

In [7]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(SentimentRNN, self).__init__()
        
        # 1. Embedding 层
        # 作用：将稀疏的整数 ID 转换为稠密的词向量
        # vocab_size: 词典大小
        # embedding_dim: 词向量的维度
        # padding_idx=0: 告诉模型 ID 为 0 的是填充项，其向量固定为 0，不参与梯度更新
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 2. RNN 层
        # 输入维度是 embedding_dim (上一层的输出)
        # hidden_size: RNN 隐藏层维度
        # batch_first=True: 输入形状为 (batch, seq, feature)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        
        # 3. 输出层 (全连接层)
        # 将 RNN 的输出映射到分类类别数 (output_size)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x: 输入 ID 序列，形状 (batch_size, seq_len)
        
        # 1. Embedding 查表
        # 将每个 ID 替换为对应的向量
        # embedded 形状: (batch_size, seq_len, embedding_dim)
        embedded = self.embedding(x)
        
        # 2. RNN 前向传播
        # rnn_out: 所有时间步的输出，形状 (batch_size, seq_len, hidden_size)
        # _: 最后一个时间步的隐藏状态 (这里不需要，用 _ 忽略)
        rnn_out, _ = self.rnn(embedded)
        
        # 3. 取最后一个时间步的输出
        # rnn_out[:, -1, :] 表示取所有样本的最后一个序列位置的特征
        # 注意：如果 padding 很多，直接取 -1 可能会取到 pad 的输出。
        # 严格来说应该取有效长度的最后一个位置，但 RNN 对 pad (输入为0) 的处理通常会让 hidden state 保持不变或变化很小，
        # 且 Embedding 中 padding_idx=0 保证了输入为 0 向量，所以这里简化处理直接取 -1。
        last_out = rnn_out[:, -1, :]
        
        # 4. 全连接分类
        # logits 形状: (batch_size, output_size)
        logits = self.fc(last_out)
        return logits

# 实例化模型参数
vocab_size = len(tokenizer) # 词典大小
embedding_dim = 32          # 词向量维度，例如用 32 维向量表示一个词
hidden_size = 16            # RNN 隐藏层维度
output_size = 2             # 二分类 (正面/负面)

# 创建模型实例并移动到设备
model_cls = SentimentRNN(vocab_size, embedding_dim, hidden_size, output_size).to(device)
print(model_cls)

SentimentRNN(
  (embedding): Embedding(45, 32, padding_idx=0)
  (rnn): RNN(32, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=2, bias=True)
)


### 4. 训练模型
使用 CrossEntropyLoss 进行分类训练。

In [8]:
# 定义优化器：使用 Adam 优化器，学习率 0.01
optimizer = optim.Adam(model_cls.parameters(), lr=0.01)
# 定义损失函数：交叉熵损失，用于多分类任务 (包含二分类)
criterion = nn.CrossEntropyLoss()

print("开始训练情感分类模型...")

# 训练循环：20 个 epoch
for epoch in range(20):
    total_loss = 0 # 记录总损失
    correct = 0    # 记录预测正确的样本数
    total = 0      # 记录总样本数
    
    # 遍历 DataLoader
    for X_batch, y_batch in dataloader:
        # 将数据移动到设备 (GPU/CPU)
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # 1. 清空梯度
        optimizer.zero_grad()
        
        # 2. 前向传播
        outputs = model_cls(X_batch)
        
        # 3. 计算损失
        loss = criterion(outputs, y_batch)
        
        # 4. 反向传播
        loss.backward()
        
        # 5. 更新参数
        optimizer.step()
        
        # 统计指标
        total_loss += loss.item() # 累加损失
        
        # 计算准确率
        # torch.argmax(outputs, dim=1) 返回每行最大值的索引，即预测类别
        pred = torch.argmax(outputs, dim=1)
        # 统计预测正确的个数
        correct += (pred == y_batch).sum().item()
        # 统计总数
        total += y_batch.size(0)
        
    # 打印当前 epoch 的平均损失和准确率
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}, Acc: {correct/total:.2f}")

开始训练情感分类模型...
Epoch 1, Loss: 1.0164, Acc: 0.50
Epoch 2, Loss: 0.8937, Acc: 0.50
Epoch 3, Loss: 0.6444, Acc: 0.38
Epoch 4, Loss: 0.6970, Acc: 0.50
Epoch 5, Loss: 0.5342, Acc: 0.75
Epoch 6, Loss: 0.5337, Acc: 0.75
Epoch 7, Loss: 0.4342, Acc: 0.88
Epoch 8, Loss: 0.4294, Acc: 0.75
Epoch 9, Loss: 0.3104, Acc: 0.88
Epoch 10, Loss: 0.3966, Acc: 0.75
Epoch 11, Loss: 0.5076, Acc: 0.75
Epoch 12, Loss: 0.2102, Acc: 0.88
Epoch 13, Loss: 0.1725, Acc: 1.00
Epoch 14, Loss: 0.0896, Acc: 1.00
Epoch 15, Loss: 0.0982, Acc: 1.00
Epoch 16, Loss: 0.0734, Acc: 1.00
Epoch 17, Loss: 0.0382, Acc: 1.00
Epoch 18, Loss: 0.0223, Acc: 1.00
Epoch 19, Loss: 0.0170, Acc: 1.00
Epoch 20, Loss: 0.0129, Acc: 1.00
